# Model Intercomparison

This notebook allows for the comparison of multiple model runs. It scans the output directories of different experiments and generates comparative plots.

## Features

*   **Metric Comparison**: Side-by-side comparison of scalar metrics (MAE, RMSE, CRPS, etc.).
*   **Vertical Profiles**: Comparison of error profiles across pressure levels.
*   **Energy Spectra**: Comparison of kinetic energy spectra.
*   **Maps**: Visual comparison of forecast fields and error maps.

## Relevant Code

The orchestration logic resides in [`run_from_config`](../src/swissclim_evaluations/intercompare.py#L217).

Key functions:
*   [`intercompare_vertical_profiles`](../src/swissclim_evaluations/intercomparison/modules/vertical_profiles.py#L71): Overlays vertical profile metrics from multiple models.
*   [`scan_model_sets`](../src/swissclim_evaluations/intercomparison/core.py#L42): Identifies common files across model output directories to ensure fair comparison.


# Model Intercomparison Demo
This notebook shows how to combine saved artifacts produced by the CLI for multiple models.

## 1. Configuration

Edit the lists below to point to the per-model evaluation output folders you want to intercompare. Optionally provide human-readable labels (must match length of `model_dirs`). If `labels` is left empty or length mismatch occurs, folder basenames are used.

Please specify the path to your configuration file below.

In [ ]:
# Define the path to the configuration file
config_path_str = "config/example_config.yaml"

In [ ]:
import yaml
from pathlib import Path
import pandas as pd
import sys
import importlib
from swissclim_evaluations.intercompare import run_from_config
from swissclim_evaluations.helpers import display_outputs

# --- 1. Configuration (via YAML) ------------------------------------------
# Resolve repo root early and ensure local package is importable
def _find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    for _ in range(10):
        if (cur / "pyproject.toml").exists() or (cur / ".git").exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    return start.resolve()

project_root = _find_repo_root(Path.cwd())
root_dir = project_root  # expose as 'root_dir' for clarity in the notebook
sys.path.insert(0, str(project_root / "src"))  # prefer local package in src/

# YAML config path
if 'config_path_str' in locals() and config_path_str:
    candidate = Path(config_path_str)
    if candidate.is_absolute():
        cfg_path = candidate
    else:
        # Try relative to cwd
        if (Path.cwd() / candidate).is_file():
            cfg_path = (Path.cwd() / candidate).resolve()
        # Try relative to parent (if running from notebooks dir)
        elif (Path.cwd().parent / candidate).is_file():
            cfg_path = (Path.cwd().parent / candidate).resolve()
        # Try relative to project root (if different from above)
        elif (project_root / candidate).is_file():
            cfg_path = (project_root / candidate).resolve()
        else:
            # Fallback to original logic or just keep candidate
            cfg_path = candidate
else:
    cfg_path = (project_root / "config/intercomparison.yaml").resolve()

print("Using intercomparison config:", cfg_path)

# Load YAML and compute derived paths (without running yet)
with open(cfg_path) as f:
    cfg = yaml.safe_load(f) or {}

# Resolve relative paths against project root for display
model_dirs = [
    (project_root / p).resolve() if not str(p).startswith("/") else Path(p)
    for p in cfg.get("models", [])
 ]
labels = cfg.get("labels") or []
if not labels or len(labels) != len(model_dirs):
    labels = [p.name for p in model_dirs]
out_root = (
    (project_root / cfg.get("output_root", "output/intercomparison")).resolve()
    if not str(cfg.get("output_root", "output/intercomparison")).startswith("/")
    else Path(cfg.get("output_root")).resolve()
)


## 2. Run Intercomparison
Runs the selected intercomparison modules based on the YAML. After the run, we render per-module results in dedicated sections: figures first, then tables.

In [ ]:
RUN_ALL = False
if RUN_ALL:
    run_from_config(run_cfg)

## 3. Maps

Visual comparison of forecast fields and error maps.
Allows for a qualitative assessment of spatial patterns and biases.

**Relevant Code:**
*   [`intercompare_maps`](../src/swissclim_evaluations/intercomparison/modules/maps.py#L35): Creates multi-panel figures showing fields from different models side-by-side.

In [ ]:
# Maps — Figures
display_outputs(out_root / "maps")

## 4. Histograms

Compares the distribution of values for each variable across models.
Useful for checking if models capture the correct range and frequency of values (e.g., extreme events).

**Relevant Code:**
*   [`intercompare_histograms`](../src/swissclim_evaluations/intercomparison/modules/histograms.py#L35): Generates side-by-side histogram plots.

In [ ]:
# Histograms — Figures & Tables
display_outputs(out_root / "histograms")

## 5. Wasserstein Distance & KDE

Compares the Wasserstein distance and Kernel Density Estimates (KDE) of the distributions.
This provides a quantitative and visual comparison of how close the model distributions are to the target.

**Relevant Code:**
*   [`intercompare_wd_kde`](../src/swissclim_evaluations/intercomparison/modules/wd_kde.py#L27): Aggregates Wasserstein distances and plots KDEs.

In [ ]:
# KDE — Figures & Tables
display_outputs(out_root / "wd_kde")

## 6. Energy Spectra

Compares the kinetic energy spectra of the models.
This analysis reveals how well the models capture atmospheric variability at different spatial scales.

**Relevant Code:**
*   [`intercompare_energy_spectra`](../src/swissclim_evaluations/intercomparison/modules/energy_spectra.py#L30): Overlays spectra from multiple models.
*   [`scan_model_sets`](../src/swissclim_evaluations/intercomparison/core.py#L42): Ensures only common files are compared.

In [ ]:
# Energy Spectra — per-lead comparison (2D surface vars: energy_spectrum_*_lead*_compare.png)
display_outputs(out_root / "energy_spectra", pattern_img="energy_spectrum_*_lead*_compare.png", pattern_csv="", exclude_pattern="_ratio")


In [ ]:
# Energy Spectra — per-lead comparison (3D pressure-level vars: energy_spectrum_*hPa*_compare.png)
display_outputs(out_root / "energy_spectra", pattern_img="energy_spectrum_*hPa*_compare.png", pattern_csv="", exclude_pattern="_ratio")


In [ ]:
# Energy Spectra — Tables (LSD combined)
display_outputs(out_root / "energy_spectra", pattern_img="", pattern_csv="*.csv")

In [ ]:
display_outputs(out_root / "energy_spectra", pattern_img="*ratio*.png", pattern_csv="")

In [ ]:
# Energy Spectra — delta spectrograms (Δ log10 energy vs lead time, one panel per model)
display_outputs(out_root / "energy_spectra", pattern_img="*spectrogram_delta_compare.png", pattern_csv="")


## 7. Vertical Profiles

Compares the vertical structure of errors (e.g., NMAE) across pressure levels.
Helps identify if models perform better or worse at specific altitudes (e.g., surface vs. upper atmosphere).

**Relevant Code:**
*   [`intercompare_vertical_profiles`](../src/swissclim_evaluations/intercomparison/modules/vertical_profiles.py#L71): Overlays error profiles from multiple models on the same plot.

In [ ]:
# Vertical Profiles — Figures & Tables
display_outputs(out_root / "vertical_profiles")

## 8. Metrics (Deterministic)

Consolidates scalar metrics (MAE, RMSE, Bias, Pearson R, FSS) into a single comparison table.
This allows for a direct ranking of model performance.

**Relevant Code:**
*   [`intercompare_deterministic_metrics`](../src/swissclim_evaluations/intercomparison/modules/deterministic.py#L20): Merges CSV files from different models into a combined DataFrame.

In [ ]:
# Deterministic Metrics vs Lead Time
display_outputs(out_root / "deterministic", pattern_img="temporal_*.png", pattern_csv="temporal_metrics_combined.csv")

In [ ]:
# Metrics — Tables
display_outputs(out_root / "deterministic", exclude_pattern="temporal_")

## 9. ETS
Quick comparison plots saved during intercompare (if any).

In [ ]:
# ETS vs Lead Time
display_outputs(out_root / "ets", pattern_img="ets_*_compare.png", pattern_csv="ets_metrics_combined.csv")

In [ ]:
# ETS — Figures & Tables
display_outputs(out_root / "ets")

## 10. Probabilistic

Compares probabilistic metrics (CRPS, PIT, Spread-Skill Ratio).
Crucial for evaluating ensemble prediction systems.

**Relevant Code:**
*   [`intercompare_probabilistic`](../src/swissclim_evaluations/intercomparison/modules/probabilistic.py#L32):
    *   Overlays PIT histograms (requires `pit_hist_*.npz` artifacts from single-model runs).
    *   Compares CRPS maps.
    *   Aggregates CRPS and SSR summary statistics.


In [ ]:
# Probabilistic Metrics vs Lead Time
display_outputs(out_root / "probabilistic", pattern_img="temporal_*.png", pattern_csv="temporal_metrics_combined.csv")

In [ ]:
# Probabilistic — Figures & Tables
display_outputs(out_root / "probabilistic", exclude_pattern=["temporal_"])